# Online and Offline Feature Parity

**Prerequisites:** Python basics and concepts 1–16 in the MLOps track. **Estimated time:** 40 minutes.

## Objectives

- Measure offline-online value mismatch and freshness as distinct contract checks.
- Explain an operational threshold and its owner response.
- Test a deterministic failure boundary.


## Concept recap and assumptions

This lesson uses synthetic local data and a transparent rule. The numeric threshold is an explicit policy example that must be calibrated to workload and SLO; it is not a universal production default. No cloud credentials or external service are used.


In [ ]:
# Load the paired dependency-light implementation from the repository root.
import runpy
impl=runpy.run_path('mlops/implementations/20-feature-parity.py')


## Baseline implementation

The implementation exposes the computation and decision separately, so metrics can be inspected before action. Read its validation checks, then change one deterministic input at a time.


In [ ]:
now='2026-01-01T00:01:00'; stamp='2026-01-01T00:00:00'
rows=[{'offline':{'value':2.0},'online':{'value':2.0,'event_time':stamp,'observed_at':now}},{'offline':{'value':3.0},'online':{'value':3.1,'event_time':stamp,'observed_at':now}}]
report=impl['audit_rows'](rows,max_mismatch_rate=.4,max_age_seconds=120)
print(report)
assert report['mismatch_rate']==.5


## Comparison and operational threshold

Compare the metric to the stated threshold and explain who responds. Do not automate model retraining from this signal alone. A real deployment should record the threshold version, observation window, sample floor and remediation owner.


In [ ]:
old={'value':1,'event_time':'2025-12-31T23:00:00','observed_at':'2026-01-01T00:00:00'}
assert not impl['compare_feature']({'value':1},old,max_age_seconds=60)['fresh']


## Failure cases and exercise

Consider a missing observation, a boundary value and a shifted input. Which should page, block a release or only open an investigation?

**Exercise:** Change one threshold and one data point. Predict the result first, then encode it as an assertion.


In [ ]:
clean={'offline':{'value':5},'online':{'value':5,'event_time':'2026-01-01T00:00:00','observed_at':'2026-01-01T00:00:30'}}
assert impl['audit_rows']([clean],max_age_seconds=60)['decision']=='pass'


## Summary and next lessons

You exercised online and offline feature parity with deterministic evidence and an explicit policy boundary. Next read the linked concept note and practice explaining the response under a realistic incident scenario.
